Misclassified samples are grouped by class from misclassified_samples_v9.txt.

Correct samples are reconstructed by scanning the dataset and excluding misclassified ones.

For each class, it saves:

classX_misclassified_v9.png (worst predictions)

classX_correct_v9.png (best predictions)

Summary saved as class_summary_v9.txt with counts per class.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os
import importlib.util
from collections import defaultdict
import matplotlib.pyplot as plt
from PIL import Image
import glob

# ===============================
# Paths
# ===============================
project_root = "/content/drive/MyDrive/deepfake-detection/microexpression"
misclassified_file = os.path.join(project_root, "reports", "misclassified_samples_v9.txt")
save_dir = os.path.join(project_root, "reports", "misclassified_plots_v9")
os.makedirs(save_dir, exist_ok=True)

# ===============================
# Load MicroExpressionDataset
# ===============================
load_seq_path = os.path.join(project_root, "load_sequences.py")

if not os.path.exists(load_seq_path):
    raise FileNotFoundError(f"❌ load_sequences.py not found at {load_seq_path}")

spec = importlib.util.spec_from_file_location("load_sequences", load_seq_path)
load_sequences = importlib.util.module_from_spec(spec)
spec.loader.exec_module(load_sequences)

MicroExpressionDataset = load_sequences.MicroExpressionDataset
print("📂 MicroExpressionDataset imported successfully!")

root_dir = "/content/drive/MyDrive/deepfake-detection/dataset/microexpression_processed"
cache_dir = os.path.join(project_root, "cache")

dataset = MicroExpressionDataset(root_dir=root_dir, seq_len=16, transform=None, cache_dir=cache_dir)
print(f"✅ Dataset loaded: {len(dataset)} samples")

# Create mapping index → image path
index_to_path = {i: dataset.samples[i][0] for i in range(len(dataset))}

# ===============================
# Read Misclassified File
# ===============================
misclassified_by_class = defaultdict(list)
correct_by_class = defaultdict(list)

print(f"📄 Reading misclassified samples from: {misclassified_file}")
with open(misclassified_file, "r") as f:
    for line in f:
        if not line.strip():
            continue
        try:
            parts = line.strip().split(",")
            index = int(parts[0].split(":")[1].strip())
            true_label = int(parts[1].split(":")[1].strip())
            pred_label = int(parts[2].split(":")[1].strip())

            image_path = index_to_path.get(index, None)
            if image_path:
                misclassified_by_class[true_label].append((image_path, true_label, pred_label))
        except Exception as e:
            print(f"⚠️ Skipping malformed line: {line.strip()} ({e})")

print(f"🔍 Misclassified sample counts: {{cls: len(samples) for cls, samples in misclassified_by_class.items()}}")

# ===============================
# Identify Correctly Classified Samples
# ===============================
all_indices = set(range(len(dataset)))
misclassified_indices = {int(parts[0].split(":")[1].strip()) for parts in
                         [line.strip().split(",") for line in open(misclassified_file) if line.strip()]}

correct_indices = all_indices - misclassified_indices
for idx in correct_indices:
    path = index_to_path[idx]
    label = dataset.samples[idx][1]
    correct_by_class[label].append((path, label, label))

def get_first_frame_path(seq_dir):
    """Return the first frame image path inside a sequence directory."""
    frame_files = sorted(glob.glob(os.path.join(seq_dir, "*.jpg")) + glob.glob(os.path.join(seq_dir, "*.png")))
    if frame_files:
        return frame_files[0]
    return None

def plot_samples(samples, title, save_path, max_images=8):
    plt.figure(figsize=(12, 6))
    for i, (seq_dir, true_lbl, pred_lbl) in enumerate(samples[:max_images]):
        frame_path = get_first_frame_path(seq_dir)
        if not frame_path:
            print(f"⚠️ No frames found in {seq_dir}")
            continue
        try:
            img = Image.open(frame_path).convert("RGB")
        except Exception as e:
            print(f"⚠️ Could not open {frame_path}: {e}")
            continue
        plt.subplot(2, 4, i + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"T:{true_lbl} P:{pred_lbl}", fontsize=8)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ===============================
# Generate Plots
# ===============================
for cls in range(6):
    # Misclassified
    if misclassified_by_class[cls]:
        plot_samples(
            misclassified_by_class[cls],
            f"Class {cls} - Misclassified",
            os.path.join(save_dir, f"class_{cls}_misclassified.png")
        )
    else:
        print(f"⚠️ No samples to plot for Class {cls} - Misclassified")

    # Correct
    if correct_by_class[cls]:
        plot_samples(
            correct_by_class[cls],
            f"Class {cls} - Correct",
            os.path.join(save_dir, f"class_{cls}_correct.png")
        )
    else:
        print(f"⚠️ No samples to plot for Class {cls} - Correct")

print(f"✅ Inspection complete. Plots saved to: {save_dir}")


📂 MicroExpressionDataset imported successfully!
📂 Initializing MicroExpressionDataset...
🗂 Cache directory set to: /content/drive/MyDrive/deepfake-detection/microexpression/cache
🔍 Scanning dataset folders...
✅ Found 220 video samples across 6 emotion categories.
✅ Dataset loaded: 220 samples
📄 Reading misclassified samples from: /content/drive/MyDrive/deepfake-detection/microexpression/reports/misclassified_samples_v9.txt
🔍 Misclassified sample counts: {cls: len(samples) for cls, samples in misclassified_by_class.items()}
⚠️ No samples to plot for Class 2 - Misclassified
⚠️ No samples to plot for Class 3 - Misclassified
✅ Inspection complete. Plots saved to: /content/drive/MyDrive/deepfake-detection/microexpression/reports/misclassified_plots_v9
